In [1]:
import polars as pl
import os

In [2]:
data_dir = "raw_data/sales_pers.purchase_history_daily_chunk_*.parquet"

purchases = pl.scan_parquet(data_dir)

In [3]:
purchases.collect().null_count()

timestamp,user_id,item_id,event_type,event_value,price,date_key,quantity,customer_id,created_date,updated_date,channel,payment,location,discount,is_deleted
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [4]:
purchases.collect_schema()

Schema([('timestamp', Int64),
        ('user_id', String),
        ('item_id', String),
        ('event_type', String),
        ('event_value', Decimal(precision=38, scale=4)),
        ('price', Decimal(precision=38, scale=4)),
        ('date_key', Int32),
        ('quantity', Int32),
        ('customer_id', Int32),
        ('created_date', Datetime(time_unit='us', time_zone=None)),
        ('updated_date', Datetime(time_unit='us', time_zone=None)),
        ('channel', String),
        ('payment', String),
        ('location', Int32),
        ('discount', Decimal(precision=38, scale=4)),
        ('is_deleted', Boolean)])

In [5]:
import polars.selectors as cs

In [6]:
purchases_string = purchases.select(cs.string())

In [7]:
with pl.Config(tbl_cols=-1, tbl_width_chars=1000):
    print(
        purchases_string.select(
            (cs.string().str.strip_chars().str.to_lowercase() == "không xác định").sum()
        ).collect()
    )

shape: (1, 5)
┌─────────┬─────────┬────────────┬─────────┬─────────┐
│ user_id ┆ item_id ┆ event_type ┆ channel ┆ payment │
│ ---     ┆ ---     ┆ ---        ┆ ---     ┆ ---     │
│ u32     ┆ u32     ┆ u32        ┆ u32     ┆ u32     │
╞═════════╪═════════╪════════════╪═════════╪═════════╡
│ 0       ┆ 0       ┆ 0          ┆ 119     ┆ 331952  │
└─────────┴─────────┴────────────┴─────────┴─────────┘


In [ ]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        purchases_string
        .filter(pl.col("event_type").is_not_null())
        .group_by("event_type")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

shape: (1, 2)
┌────────────┬──────────┐
│ event_type ┆ count    │
│ ---        ┆ ---      │
│ str        ┆ u32      │
╞════════════╪══════════╡
│ Purchase   ┆ 35729825 │
└────────────┴──────────┘


In [19]:
purchases = purchases.drop(["event_type"])

In [9]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        purchases_string
        .filter(pl.col("channel").is_not_null())
        .group_by("channel")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

shape: (11, 2)
┌────────────────┬──────────┐
│ channel        ┆ count    │
│ ---            ┆ ---      │
│ str            ┆ u32      │
╞════════════════╪══════════╡
│ In-Store       ┆ 33016358 │
│ iOS            ┆ 1397128  │
│ SPE            ┆ 602896   │
│ Android        ┆ 483885   │
│ Web            ┆ 146401   │
│ Call           ┆ 63924    │
│ CRM Partner    ┆ 16640    │
│ Chat           ┆ 2310     │
│ Wholesale      ┆ 162      │
│ Không xác định ┆ 119      │
│ TKS            ┆ 2        │
└────────────────┴──────────┘


In [10]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        purchases_string
        .filter(pl.col("payment").is_not_null())
        .group_by("payment")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

shape: (10, 2)
┌────────────────┬──────────┐
│ payment        ┆ count    │
│ ---            ┆ ---      │
│ str            ┆ u32      │
╞════════════════╪══════════╡
│ Tiền mặt       ┆ 22454572 │
│ VietQR         ┆ 5766946  │
│ Cà thẻ         ┆ 4040302  │
│ VNPay          ┆ 2080460  │
│ MoMo           ┆ 747996   │
│ Không xác định ┆ 331952   │
│ ZaloPay        ┆ 242983   │
│ ShopeePay      ┆ 45508    │
│ Kredivo        ┆ 19044    │
│ Chuyển khoản   ┆ 62       │
└────────────────┴──────────┘


In [17]:
purchases = purchases.with_columns(
    payment=(
        pl.when(pl.col("payment").str.strip_chars().str.to_lowercase() == "không xác định")
        .then(pl.lit("Chuyển khoản"))
        .otherwise(pl.col("payment"))
    )
)
purchases_string = purchases_string.with_columns(
    payment=(
        pl.when(pl.col("payment").str.strip_chars().str.to_lowercase() == "không xác định")
        .then(pl.lit("Chuyển khoản"))
        .otherwise(pl.col("payment"))
    )
)

In [18]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        purchases_string
        .filter(pl.col("payment").is_not_null())
        .group_by("payment")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

shape: (9, 2)
┌──────────────┬──────────┐
│ payment      ┆ count    │
│ ---          ┆ ---      │
│ str          ┆ u32      │
╞══════════════╪══════════╡
│ Tiền mặt     ┆ 22454572 │
│ VietQR       ┆ 5766946  │
│ Cà thẻ       ┆ 4040302  │
│ VNPay        ┆ 2080460  │
│ MoMo         ┆ 747996   │
│ Chuyển khoản ┆ 332014   │
│ ZaloPay      ┆ 242983   │
│ ShopeePay    ┆ 45508    │
│ Kredivo      ┆ 19044    │
└──────────────┴──────────┘


In [11]:
purchases_num = purchases.select(cs.numeric())

In [12]:
with pl.Config(tbl_cols=-1, tbl_width_chars=1000):
    print(
        purchases_num.select(
            (cs.numeric().is_null()).sum()
        ).collect()
    )

shape: (1, 8)
┌───────────┬─────────────┬───────┬──────────┬──────────┬─────────────┬──────────┬──────────┐
│ timestamp ┆ event_value ┆ price ┆ date_key ┆ quantity ┆ customer_id ┆ location ┆ discount │
│ ---       ┆ ---         ┆ ---   ┆ ---      ┆ ---      ┆ ---         ┆ ---      ┆ ---      │
│ u32       ┆ u32         ┆ u32   ┆ u32      ┆ u32      ┆ u32         ┆ u32      ┆ u32      │
╞═══════════╪═════════════╪═══════╪══════════╪══════════╪═════════════╪══════════╪══════════╡
│ 0         ┆ 0           ┆ 0     ┆ 0        ┆ 0        ┆ 0           ┆ 0        ┆ 0        │
└───────────┴─────────────┴───────┴──────────┴──────────┴─────────────┴──────────┴──────────┘


In [20]:
purchases_bool = purchases.select(cs.boolean())

In [21]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        purchases_bool
        .filter(pl.col("is_deleted").is_not_null())
        .group_by("is_deleted")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

shape: (1, 2)
┌────────────┬──────────┐
│ is_deleted ┆ count    │
│ ---        ┆ ---      │
│ bool       ┆ u32      │
╞════════════╪══════════╡
│ false      ┆ 35729825 │
└────────────┴──────────┘


In [22]:
purchases = purchases.drop(["is_deleted"])

In [ ]:
clean_dir = "clean_data"
os.makedirs(clean_dir, exist_ok=True)
output_path = os.path.join(clean_dir, "purchases_cleaned.parquet")

purchases.sink_parquet(output_path, compression="zstd")

print(f"🔢 Số dòng:    {purchases.select(pl.len()).collect().item():,} dòng")

✅ Xuất thành công purchases_cleaned sau: 8.19 giây!
📁 Đường dẫn:  clean_data\purchases_cleaned.parquet
📦 Dung lượng: 1620.13 MB
🔢 Số dòng:    35,729,825 dòng
